# CS383 — Assignment 1
## Joining and Cleaning Restaurant Inspection Data

*Due: see LMS*

### Scenario
You've been given access to a small relational database of NYC restaurant inspection records, split across three tables:

- `restaurants` — one row per establishment
- `inspections` — one row per inspection visit
- `violations` — one row per violation cited during an inspection

Your task is to write SQL queries that join across these tables, then move into Pandas to clean and summarize what you find.

**This is the first time you're seeing this specific database.** That's intentional — use what you learned about `SELECT`, `WHERE`, `GROUP BY`, joins, and schema-checking (`PRAGMA table_info`, `.dtypes`, `.head()`) to explore it yourself before answering the required questions below. Nothing here has been walked through in class.

---

### Getting started

Run the cell below once. It builds the three tables above from live NYC Open Data if there's a network connection available, or realistic sample data if not. **Do not skip or modify this cell** — everything after it assumes these three tables exist in `restaurant_inspections.db`.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import requests

SOCRATA_URL = "https://data.cityofnewyork.us/resource/43nn-pn8j.json"

try:
    response = requests.get(
        SOCRATA_URL,
        params={
            "$limit": 15000,
            "$order": "inspection_date DESC",
            "$select": "camis,dba,boro,cuisine_description,inspection_date,action,"
                        "score,grade,violation_code,violation_description,critical_flag",
        },
        timeout=10,
    )
    response.raise_for_status()
    raw = pd.DataFrame(response.json())
    if "score" in raw.columns:
        raw["score"] = pd.to_numeric(raw["score"], errors="coerce")

    restaurants = raw[["camis", "dba", "boro", "cuisine_description"]].drop_duplicates(
        subset="camis"
    ).reset_index(drop=True)

    inspections = raw[["camis", "inspection_date", "action", "score", "grade"]].drop_duplicates(
        subset=["camis", "inspection_date"]
    ).reset_index(drop=True)
    inspections.insert(0, "inspection_id", range(1, len(inspections) + 1))

    with_ids = raw.merge(
        inspections[["camis", "inspection_date", "inspection_id"]], on=["camis", "inspection_date"]
    )
    violations = with_ids.loc[
        with_ids["violation_code"].notna(),
        ["inspection_id", "violation_code", "violation_description", "critical_flag"],
    ].reset_index(drop=True)

    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)

    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza",
                       "Japanese", "Caribbean", "Bakery", "Coffee/Tea", "Chicken"]

    n_restaurants = 150
    camis_ids = np.arange(50_000_001, 50_000_001 + n_restaurants)
    restaurants = pd.DataFrame({
        "camis": camis_ids,
        "dba": [f"Restaurant {i + 1}" for i in range(n_restaurants)],
        "boro": rng.choice(boroughs_list, size=n_restaurants, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n_restaurants),
    })

    # Deliberately messy cuisine text on a chunk of rows (inconsistent case/whitespace).
    messy_idx = rng.choice(restaurants.index, size=20, replace=False)
    def _messify(s):
        pick = rng.integers(0, 3)
        if pick == 0:
            return s.upper()
        elif pick == 1:
            return f"  {s.lower()}  "
        return s.lower()
    restaurants.loc[messy_idx, "cuisine_description"] = restaurants.loc[
        messy_idx, "cuisine_description"
    ].apply(_messify)

    # A handful of exact duplicate restaurant rows.
    restaurants = pd.concat(
        [restaurants, restaurants.sample(5, random_state=1)], ignore_index=True
    )

    n_inspections = 400
    inspections = pd.DataFrame({
        "inspection_id": np.arange(1, n_inspections + 1),
        "camis": rng.choice(camis_ids, size=n_inspections),
        "inspection_date": pd.date_range("2025-01-01", periods=n_inspections, freq="18h").astype(str),
        "action": "Violations were cited in the following area(s).",
        "score": rng.integers(0, 60, size=n_inspections),
        "grade": rng.choice(["A", "B", "C", None], size=n_inspections, p=[0.55, 0.25, 0.10, 0.10]),
    })
    # A few duplicate inspection rows too.
    inspections = pd.concat(
        [inspections, inspections.sample(4, random_state=2)], ignore_index=True
    )

    violation_pool = [
        ("04L", "Evidence of mice or live mice present", "Critical"),
        ("06C", "Food not protected from potential source of contamination", "Critical"),
        ("08A", "Facility not vermin proof", "Not Critical"),
        ("10F", "Non-food contact surface improperly constructed", "Not Critical"),
        ("02B", "Hot food item not held at or above 140 F", "Critical"),
        ("04N", "Filth flies or food/refuse/sewage-associated flies present", "Critical"),
        ("06D", "Food contact surface not properly washed", "Critical"),
        ("05D", "Hand washing facility not accessible", "Critical"),
    ]
    n_violations = 700
    picks = rng.integers(0, len(violation_pool), size=n_violations)
    violations = pd.DataFrame({
        "inspection_id": rng.choice(inspections["inspection_id"].unique(), size=n_violations),
        "violation_code": [violation_pool[i][0] for i in picks],
        "violation_description": [violation_pool[i][1] for i in picks],
        "critical_flag": [violation_pool[i][2] for i in picks],
    })

    live = False

print(f"{'Live' if live else 'Offline fallback'} data loaded.")
print(f"restaurants: {len(restaurants):,} rows | inspections: {len(inspections):,} rows | violations: {len(violations):,} rows")

conn = sqlite3.connect("restaurant_inspections.db")
restaurants.to_sql("restaurants", conn, if_exists="replace", index=False)
inspections.to_sql("inspections", conn, if_exists="replace", index=False)
violations.to_sql("violations", conn, if_exists="replace", index=False)

print("Tables ready in restaurant_inspections.db: restaurants, inspections, violations")

---

### Explore before you answer

Before jumping into the required questions, take a few minutes to look around:

- What columns does each table have, and what do they look like? (`PRAGMA table_info(...)`, `.dtypes`, `.head()`)
- How do the three tables relate to each other — which columns would you join on?
- Do you notice anything odd or messy in the data as you look at it?

Use the scratch cell below (and add more cells if you want) to explore. This part isn't graded directly, but it'll make the required questions much easier.

In [ ]:
# Scratch space -- explore restaurants, inspections, and violations here.


---

## Required Deliverable 1 — Restaurants Per Cuisine, Cleaned

The `cuisine_description` column has inconsistent capitalization and stray whitespace in places, and the `restaurants` table has a few exact duplicate rows.

Clean both issues in Pandas, then produce a count of restaurants per (cleaned) cuisine type, sorted from most to fewest.

In [ ]:
# Your code here.


---

## Required Deliverable 2 — Worst Inspection Scores, With Restaurant Names (JOIN required)

Using a SQL `JOIN` across `restaurants` and `inspections`, find the 10 individual inspections with the **highest** score (a higher score means more violations) among inspections that also received a letter grade.

Your result should include: restaurant name, borough, cuisine, inspection date, score, and grade.

In [ ]:
# Your code here.


---

## Required Deliverable 3 — Most Common Critical Violations, By Cuisine (JOIN required, all 3 tables)

Pick one cuisine type. Using SQL `JOIN`s across all three tables, find the 5 most common `"Critical"` violation descriptions among restaurants of that cuisine.

Then reproduce the same result using Pandas `.merge()` instead of SQL, and confirm the two approaches agree.

In [ ]:
# SQL version -- your code here.


In [ ]:
# Pandas version -- your code here.


In [ ]:
# Confirm the SQL and Pandas results agree -- your code here.


---

## Required Deliverable 4 — Handle the Missing Grades

Not every inspection has a letter grade — some are still missing (`None` / `NaN`).

Decide how to handle this for the purpose of computing an **average score by grade**: drop the missing rows? Label them `"Ungraded"` and keep them separate? Something else? Write one or two sentences explaining your reasoning *before* your code, then implement it.

**Your reasoning:**



In [ ]:
# Your code here.


---

## Required Deliverable 5 — Summary for a Non-Technical Reader

Pick one finding from your analysis above that an actual restaurant-goer (not a data scientist) would care about.

Produce one clean summary table or chart, and write a two-to-three sentence explanation in plain language — no jargon, no code terms. Imagine you're telling a friend, not writing a lab report.

In [ ]:
# Your code here.


**Your summary (plain language):**



---

## Grading Rubric

| Criterion | Points |
|---|---|
| Deliverable 1 — cleaning is correct and counts are right | 15 |
| Deliverable 2 — JOIN is correct and produces the right 10 rows | 20 |
| Deliverable 3 — 3-table JOIN correct; SQL and Pandas results genuinely match | 25 |
| Deliverable 4 — reasoning is stated and implementation matches it | 15 |
| Deliverable 5 — finding is genuinely non-technical, table/chart is clear | 15 |
| Notebook runs top to bottom without errors | 10 |
| **Total** | **100** |

---

## Submission

Commit and push your completed notebook to your assignment repository:

```bash
git add -A
git commit -m "Assignment 1: restaurant inspections"
git push
```

Make sure your notebook runs top to bottom without errors before you submit — a notebook that only works because cells were run out of order will not be graded as passing.